In [7]:
import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")

# Keep only gap-risk-clean stocks, take the top 15
clean_candidates = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
top_15 = clean_candidates.head(15)

# Always include NIFTY and BANKNIFTY, gap-risk-checked the same way
must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")]

final_candidates = pd.concat([top_15, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)
final_candidates

,symbol,credit_spread_score,gap_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr)
0,NESTLEIND,90.7,NaN,54,24.2,59.3,2.72,5.96,4.37
1,ITC,86.0,NaN,51,24.2,60.8,2.48,9.56,4.68
2,SUNPHARMA,80.8,NaN,52,20.0,67.3,2.66,7.31,3.53
3,HAVELLS,80.1,NaN,60,21.6,66.7,3.27,6.58,4.42
4,DRREDDY,79.7,NaN,57,22.2,68.4,2.84,7.88,4.27
5,ULTRACEMCO,79.4,NaN,66,19.0,66.7,2.39,7.21,3.61
6,COLPAL,79.3,NaN,58,19.2,67.2,3.10,5.84,2.32
7,PIIND,78.2,NaN,49,22.2,63.3,3.55,8.76,4.07
8,TECHM,78.1,NaN,64,19.6,62.5,3.16,6.85,4.37
9,ICICIPRULI,77.9,NaN,58,21.6,65.5,3.18,8.00,4.94


In [8]:
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.fetch_data import fetch_historical_data, save_raw_data
from src.supertrend import apply_supertrend
from src.flip_analysis import detect_flips
from src.monthly_range import compute_monthly_range
from src.summary_stats import compute_summary_stats
from src.export_report import export_full_report
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        df = fetch_historical_data(kite, token, from_date=datetime(2022, 1, 1), to_date=datetime.now(), interval="4hour")
        save_raw_data(df, symbol)
        df = df.set_index("date")

        df_st = apply_supertrend(df)
        trades = detect_flips(df_st)
        monthly = compute_monthly_range(df)
        summary = compute_summary_stats(trades, monthly)
        export_full_report(trades, monthly, summary, symbol)

        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-01...
Saved 2258 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/17] NESTLEIND — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-01...
Saved 2258 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ITC_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/ITC_full_report.xlsx
[2/17] ITC — refreshed
Fetching 2022-01-01 to 2023-01-31...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-01...
Saved 2258 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/SUNPHARMA_full_report.xlsx
[3/17] SUNPHARMA — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...


...

/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...
Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-01...
Saved 2258 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BANKNIFTY_full_report.xlsx
[16/17] BANKNIFTY — refreshed
Fetching 2022-01-01 to 2023-01-31...
Fetching 2023-02-01 to 2024-03-02...


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


Fetching 2024-03-03 to 2025-04-02...
Fetching 2025-04-03 to 2026-05-03...
Fetching 2026-05-04 to 2026-08-01...
Saved 2258 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_4hour.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/NIFTY_full_report.xlsx
[17/17] NIFTY — refreshed


/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/notebooks/../src/monthly_range.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["year_month"] = df.index.to_period("M")


In [9]:
from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)
status.sort_values("trend_age_days")

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct
14,TORNTPHARM,UP,1,23.1,False,False
13,BHARTIARTL,UP,1,20.2,False,False
16,NIFTY,UP,3,15.2,False,False
1,ITC,UP,5,23.7,False,False
3,HAVELLS,UP,8,21.6,False,False
7,PIIND,UP,12,22.1,False,False
10,DABUR,DOWN,18,19.2,False,False
9,ICICIPRULI,UP,22,21.6,True,False
4,DRREDDY,DOWN,23,22.2,True,False
15,BANKNIFTY,DOWN,24,19.0,True,False
